# CartIQ — Phase 5: GRU-based Sequential Recommender
**CS Machine Learning Course Project — Group 3**

Treats each user's chronological purchase history as a sequence fed into a GRU network to capture weekly/monthly purchasing rhythms.

In [ ]:
!pip install -q torch scikit-learn matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, time, warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
sns.set_style('whitegrid')

In [ ]:
# Load data
prior = pd.read_parquet('data/prior.parquet')
orders = pd.read_parquet('data/orders.parquet')
train_labels = pd.read_parquet('data/train_labels.parquet')
user_sequences = pd.read_parquet('data/user_sequences.parquet')

# Product ID mapping
all_product_ids = prior['product_id'].unique()
prod2idx = {pid: i + 1 for i, pid in enumerate(all_product_ids)}  # 0 = padding
n_products = len(prod2idx) + 1  # +1 for padding token

MAX_SEQ_LEN = 50

print(f'Unique products: {n_products - 1:,}')
print(f'Users with sequences: {len(user_sequences):,}')

## 5.1 Sequence Dataset
For each user: `(input_sequence, next_basket_labels)` where the target is a multi-hot vector of products in their train order.

In [ ]:
# Build target: for each user, which products appear in their train order?
train_users_products = train_labels.groupby('user_id')['product_id'].apply(set).to_dict()

class GRUDataset(Dataset):
    def __init__(self, user_ids, user_sequences_df, train_targets, prod2idx, max_len, top_k_products):
        self.samples = []
        self.top_k = top_k_products  # Only predict on top-K products for efficiency
        
        for uid in user_ids:
            row = user_sequences_df[user_sequences_df['user_id'] == uid]
            if len(row) == 0:
                continue
            
            seq = row.iloc[0]['sequence']
            # Convert to indices
            seq_idx = [prod2idx.get(p, 0) for p in seq[-max_len:]]
            
            # Target: which of top_k products did user reorder?
            target_set = train_targets.get(uid, set())
            target = np.zeros(len(top_k_products), dtype=np.float32)
            for i, pid in enumerate(top_k_products):
                if pid in target_set:
                    target[i] = 1.0
            
            self.samples.append((seq_idx, target))
        
        print(f'  Built {len(self.samples)} sequences')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        seq, target = self.samples[idx]
        return torch.tensor(seq, dtype=torch.long), torch.tensor(target, dtype=torch.float32)

def collate_fn(batch):
    seqs, targets = zip(*batch)
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    targets = torch.stack(targets)
    return padded, lengths, targets

# Use top 2000 most-ordered products for output layer
top_products = prior['product_id'].value_counts().head(2000).index.tolist()
print(f'Predicting on top {len(top_products)} products')

# Split users
from sklearn.model_selection import train_test_split

all_seq_users = user_sequences['user_id'].unique()
train_users_set = set(train_labels['user_id'].unique())
valid_users = [u for u in all_seq_users if u in train_users_set]

gru_train_users, gru_test_users = train_test_split(valid_users, test_size=0.2, random_state=42)
gru_train_users, gru_val_users = train_test_split(gru_train_users, test_size=0.125, random_state=42)

print(f'\nTrain users: {len(gru_train_users):,} | Val: {len(gru_val_users):,} | Test: {len(gru_test_users):,}')

print('\nBuilding train set:')
train_ds = GRUDataset(gru_train_users, user_sequences, train_users_products, prod2idx, MAX_SEQ_LEN, top_products)
print('Building val set:')
val_ds = GRUDataset(gru_val_users, user_sequences, train_users_products, prod2idx, MAX_SEQ_LEN, top_products)
print('Building test set:')
test_ds = GRUDataset(gru_test_users, user_sequences, train_users_products, prod2idx, MAX_SEQ_LEN, top_products)

train_loader = DataLoader(train_ds, batch_size=512, shuffle=True, collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=512, shuffle=False, collate_fn=collate_fn, num_workers=0)
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False, collate_fn=collate_fn, num_workers=0)

## 5.2 GRU Model
`Embedding(product, 64) → GRU(hidden=128, layers=2) → Linear → Sigmoid`

In [ ]:
class GRURecommender(nn.Module):
    def __init__(self, n_products, embed_dim=64, hidden_dim=128, n_layers=2, output_size=2000, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(n_products, embed_dim, padding_idx=0)
        self.gru = nn.GRU(
            embed_dim, hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
        )
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, output_size),
        )
        nn.init.xavier_uniform_(self.embedding.weight)
    
    def forward(self, seq, lengths):
        x = self.embedding(seq)
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu().clamp(min=1), batch_first=True, enforce_sorted=False
        )
        _, hidden = self.gru(packed)
        # Use last layer's hidden state
        out = hidden[-1]  # (batch, hidden_dim)
        return self.fc(out)

gru_model = GRURecommender(
    n_products=n_products,
    embed_dim=64,
    hidden_dim=128,
    n_layers=2,
    output_size=len(top_products),
    dropout=0.3,
).to(device)

total_params = sum(p.numel() for p in gru_model.parameters())
print(f'GRU Model — {total_params:,} parameters')
print(gru_model)

## 5.3 Training

In [ ]:
optimizer = torch.optim.Adam(gru_model.parameters(), lr=5e-4)
criterion = nn.BCEWithLogitsLoss()

EPOCHS = 15
PATIENCE = 3
best_val_f1 = 0
patience_counter = 0
gru_history = {'train_loss': [], 'val_loss': [], 'val_f1': []}

print(f'Training GRU for up to {EPOCHS} epochs...\n')
t0 = time.time()

for epoch in range(EPOCHS):
    gru_model.train()
    train_losses = []
    for seqs, lengths, targets in train_loader:
        seqs, targets = seqs.to(device), targets.to(device)
        optimizer.zero_grad()
        logits = gru_model(seqs, lengths)
        loss = criterion(logits, targets)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gru_model.parameters(), max_norm=1.0)
        optimizer.step()
        train_losses.append(loss.item())
    
    avg_train_loss = np.mean(train_losses)
    
    # Validate
    gru_model.eval()
    val_losses, all_preds, all_targets = [], [], []
    with torch.no_grad():
        for seqs, lengths, targets in val_loader:
            seqs, targets = seqs.to(device), targets.to(device)
            logits = gru_model(seqs, lengths)
            val_losses.append(criterion(logits, targets).item())
            all_preds.append(torch.sigmoid(logits).cpu().numpy())
            all_targets.append(targets.cpu().numpy())
    
    val_preds = np.concatenate(all_preds)
    val_targets = np.concatenate(all_targets)
    avg_val_loss = np.mean(val_losses)
    
    # Per-user Precision@10, Recall@10
    val_binary = (val_preds >= 0.5).astype(int)
    val_f1 = f1_score(val_targets.ravel(), val_binary.ravel(), zero_division=0)
    
    gru_history['train_loss'].append(avg_train_loss)
    gru_history['val_loss'].append(avg_val_loss)
    gru_history['val_f1'].append(val_f1)
    
    print(f'Epoch {epoch+1:2d}/{EPOCHS} — train_loss: {avg_train_loss:.4f} | val_loss: {avg_val_loss:.4f} | val_F1: {val_f1:.4f}')
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        torch.save(gru_model.state_dict(), 'models/gru_best.pt')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

print(f'\nTraining completed in {time.time() - t0:.1f}s')

## 5.4 Evaluate — Precision@10 and Recall@10

In [ ]:
gru_model.load_state_dict(torch.load('models/gru_best.pt', map_location=device))
gru_model.eval()

all_preds, all_targets = [], []
with torch.no_grad():
    for seqs, lengths, targets in test_loader:
        seqs = seqs.to(device)
        logits = gru_model(seqs, lengths)
        all_preds.append(torch.sigmoid(logits).cpu().numpy())
        all_targets.append(targets.numpy())

test_preds = np.concatenate(all_preds)
test_targets = np.concatenate(all_targets)

# Per-user Precision@10 and Recall@10
K = 10
precisions_at_k = []
recalls_at_k = []

for i in range(len(test_preds)):
    scores = test_preds[i]
    truth = test_targets[i]
    
    # Top K predicted products
    top_k_idx = np.argsort(scores)[::-1][:K]
    
    n_relevant = truth.sum()
    if n_relevant == 0:
        continue
    
    hits = truth[top_k_idx].sum()
    precisions_at_k.append(hits / K)
    recalls_at_k.append(hits / n_relevant)

avg_p10 = np.mean(precisions_at_k)
avg_r10 = np.mean(recalls_at_k)

# Also compute flat F1 and AUC
flat_pred = (test_preds >= 0.5).astype(int)
gru_f1 = f1_score(test_targets.ravel(), flat_pred.ravel(), zero_division=0)
gru_auc = roc_auc_score(test_targets.ravel(), test_preds.ravel())

gru_metrics = {
    'model': 'GRU Sequential',
    'precision': precision_score(test_targets.ravel(), flat_pred.ravel(), zero_division=0),
    'recall': recall_score(test_targets.ravel(), flat_pred.ravel(), zero_division=0),
    'f1': gru_f1,
    'roc_auc': gru_auc,
    'precision_at_10': avg_p10,
    'recall_at_10': avg_r10,
}

print('=== GRU Sequential Recommender ===')
print(f'Precision:    {gru_metrics["precision"]:.4f}')
print(f'Recall:       {gru_metrics["recall"]:.4f}')
print(f'F1-Score:     {gru_metrics["f1"]:.4f}')
print(f'ROC-AUC:      {gru_metrics["roc_auc"]:.4f}')
print(f'Precision@10: {avg_p10:.4f}')
print(f'Recall@10:    {avg_r10:.4f}')

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(gru_history['train_loss'], label='Train', color='#3b82f6')
axes[0].plot(gru_history['val_loss'], label='Val', color='#ef4444')
axes[0].set_title('GRU Loss', fontweight='bold')
axes[0].legend()
axes[0].set_xlabel('Epoch')

axes[1].plot(gru_history['val_f1'], color='#10b981')
axes[1].set_title('GRU Validation F1', fontweight='bold')
axes[1].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('gru_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
pd.DataFrame([gru_metrics]).to_csv('models/metrics_gru.csv', index=False)
print('GRU metrics saved.')